<a href="https://colab.research.google.com/github/salinela/carbon-portfolio-project-v2/blob/main/notebooks/07_feature_engineering.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# NB07 · Feature Engineering

Builds the 41-signal price/technical feature panel from `carbon.db`, samples to the
month-end modelling grid, and persists it (plus a 1-month-forward-return label for EDA)
to `data/processed/`.

All feature logic lives in `src/feature_engineering.py`; this notebook only orchestrates.
Run top to bottom.

### 1 · Bootstrap

In [1]:
import sys
import sqlite3
import time
from pathlib import Path
import pandas as pd
import numpy as np

In [2]:
# mount drive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# clone repo (fresh) + set identity
!git clone https://github.com/salinela/carbon-portfolio-project-v2.git /content/repo
%cd /content/repo

Cloning into '/content/repo'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 135 (delta 72), reused 95 (delta 40), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 287.87 KiB | 17.99 MiB/s, done.
Resolving deltas: 100% (72/72), done.
/content/repo


In [ ]:
!git config user.email "you@example.com"
!git config user.name "you"
!git checkout main && git pull origin main

In [ ]:
sys.path.insert(0, "/content/repo/src")
import feature_engineering as fe

In [ ]:
DRIVE = "/content/drive/MyDrive/carbon_project_v2"

# copy DB to local disk (only if you rebuild; not needed to just edit)
import os
if not os.path.exists("/content/carbon.db"):
    !cp "{DRIVE}/carbon.db" /content/carbon.db
con = sqlite3.connect("/content/carbon.db")
con.execute("PRAGMA foreign_keys = ON;")
print("fe MIN_PERIODS_FRAC:", fe.MIN_PERIODS_FRAC, "| batched:", hasattr(fe,"build_price_features_batched"))

### 2 · Confirm the source tables the build reads are populated

In [ ]:
print("quick_check:", con.execute("PRAGMA quick_check;").fetchone()[0])
for t in ["prices","corporate_actions","fx_rates","market_factors"]:
    n = con.execute(f"SELECT COUNT(*) FROM {t}").fetchone()[0]
    lo, hi = con.execute(f"SELECT MIN(date),MAX(date) FROM {t}").fetchone()
    print(f"{t:18s} {n:>11,}  {lo} … {hi}")

pd.read_sql("PRAGMA table_info(symbol_coverage);", con)[["name"]]

quick_check: ok
prices              15,654,648  2013-01-01 … 2026-06-29
corporate_actions       79,022  1972-07-28 … 2026-08-14
fx_rates                52,689  2013-01-01 … 2026-06-29
market_factors         135,300  2012-01-03 … 2026-08-26


### 3 · Confirm the `feature_engineering` module version

In [ ]:
import importlib; importlib.reload(fe)
print("MIN_PERIODS_FRAC:", fe.MIN_PERIODS_FRAC)          # expect 0.7
print("has batched     :", hasattr(fe, "build_price_features_batched"))
print("has label fn    :", hasattr(fe, "forward_return_label"))

### 4 · Full universe build (batched, memory-safe)

`build_price_features_batched` processes companies in chunks so peak RAM ≈ one batch, not
the whole universe. Lower `batch_size` to 300 if RAM is tight; raise to ~1000 on a high-RAM
machine. Output is identical to the un-batched build (verified). Expect ~15–25 min.

In [ ]:
t0 = time.time()
month_end = fe.build_price_features(con, start="2013-01-01", resample='M')
print(f"\n{month_end['signal_name'].nunique()} signals · {len(month_end):,} rows · {(time.time()-t0)/60:.1f} min")
assert month_end["signal_name"].nunique() == 41

### 5 · Panel health check before EDA

In [ ]:
month_end = pd.read_parquet(f"{DRIVE}/features_month_end.parquet")
panel = month_end.pivot_table(index=["company_id","date"], columns="signal_name", values="value")
print("panel:", panel.shape)
print("any inf:", bool(np.isinf(panel.to_numpy()).any()))
display(panel.isna().mean().sort_values(ascending=False).head(8))
panel.describe().T[["mean","std","min","max"]]

### 6. Filtering Panels

In [ ]:
# option 1 — drop by eligibility (keep only eligible firms)
panel = fe.filter_panel(panel, keep_ids=fe.eligible_ids(con))

# option 2 — drop firms with too many NAs
panel = fe.filter_panel(panel, max_missing_frac=0.5)

# option 3 — drop specific firms
panel = fe.filter_panel(panel, drop_ids=["ISIN1","ISIN2"])

# option 4 — reproduce your old Option B (drop any inf firm) — on by default
panel = fe.filter_panel(panel)

### 7 · Persist the feature panel

In [ ]:
proc = REPO / "data" / "processed"
proc.mkdir(parents=True, exist_ok=True)
feat_path = proc / "features_month_end.parquet"
month_end.to_parquet(feat_path, index=False)      # needs pyarrow; else .to_pickle(feat_path.with_suffix('.pkl'))
print("saved:", feat_path, f"({feat_path.stat().st_size/1e6:.1f} MB)")

### 8 · Persist the EDA label

1-month-forward **log** return on the same month-end grid — the target for the
feature-vs-target parts of EDA. The production label (total return + CV embargo) is built
later in `modeling.py`; this is exploratory only.

In [ ]:
label = fe.forward_return_label(con, horizon=1, kind="log")
label_path = proc / "label_fwd_return.parquet"
label.to_parquet(label_path, index=False)
print(label.shape, "->", label_path)
label.head()